In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
DATA_DIR = Path("data")
NORMALISATION = 100  # matches columns like gravity_*_per_100

def compile_city_rankings(data_dir=DATA_DIR, normalisation=NORMALISATION):
    rows = []

    for fp in sorted(data_dir.glob("*_scores.geojson")):
        city = fp.stem.replace("_scores", "").replace("-", " ").replace("_", " ").title()

        gdf = gpd.read_file(fp)

        # per-capita gravity columns preferred
        gravity_pc_cols = [
            c for c in gdf.columns
            if c.startswith("gravity_") and c.endswith(f"_per_{normalisation}")
        ]

        # fallback in case the weighted columns are not present
        if not gravity_pc_cols:
            gravity_pc_cols = [
                c for c in gdf.columns
                if c.startswith("gravity_") and "_per_" not in c and c != "avg_gravity"
            ]

        diversity_cols = [
            c for c in ["shannon", "kl_divergence", "jsd", "city_similarity"]
            if c in gdf.columns
        ]

        row = {
            "city": city,
            "n_hexes": len(gdf)
        }

        if "population" in gdf.columns:
            row["population_total"] = pd.to_numeric(
                gdf["population"], errors="coerce"
            ).fillna(0).sum()

        # category-level average per-capita accessibility
        cat_means = []
        for col in gravity_pc_cols:
            vals = pd.to_numeric(gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            mean_val = vals.fillna(0).mean()
            row[col] = mean_val
            cat_means.append(mean_val)

        # overall average across amenity categories
        row["overall_avg_accessibility"] = np.mean(cat_means) if cat_means else np.nan

        # diversity metrics
        for col in diversity_cols:
            vals = pd.to_numeric(gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            row[col] = vals.mean()

        rows.append(row)

    compiled_df = pd.DataFrame(rows)

    # optional: nicer city ordering
    compiled_df = compiled_df.sort_values("city").reset_index(drop=True)

    return compiled_df

compiled_df = compile_city_rankings()
compiled_df

,city,n_hexes,population_total,gravity_community_per_100,gravity_education_per_100,gravity_essential_services_per_100,gravity_food_and_drink_per_100,gravity_healthcare_per_100,gravity_shelter_per_100,overall_avg_accessibility,shannon,kl_divergence,jsd,city_similarity
0,Bratislava,918,7.328475e+05,1462.415111,3521.977492,396.439180,260899.325330,1112.045297,4646.653152,45339.809260,0.442729,1.039902,0.483039,0.516961
1,Brisbane,1878,9.582931e+05,760.442667,984.243458,13522.703384,1036.407136,48.732604,1622.653429,2995.863780,0.769924,0.741941,0.389326,0.610674
2,Copenhagen,1519,3.232337e+06,19.240471,62.303905,15.709945,179.959623,66.827569,9.244578,58.881015,0.741781,0.764635,0.357146,0.642854
3,Milan,512,1.546693e+06,2.745071,2.205001,4.224213,11.686548,4.199016,2.136136,4.532664,0.996371,0.509854,0.274119,0.725881
4,Singapore,1423,4.421302e+06,41.453970,32.126624,76.601005,560.033553,54.619629,45.321272,135.026009,0.704631,0.571519,0.316054,0.683946
5,Tallinn,404,4.021769e+05,121.308105,361.803448,592.926521,1263.686579,296.674042,1848.508217,747.484485,0.931498,0.471623,0.300478,0.699522
6,Turin,546,9.785185e+05,1582.861840,4121.333958,1964.697612,26832.552576,3029.855925,132.088752,6277.231777,0.829727,0.642900,0.327594,0.672406


In [3]:
# cities ranked by overall accessibility
overall_access_rank = (
    compiled_df[["city", "overall_avg_accessibility"]]
    .sort_values("overall_avg_accessibility", ascending=False)
    .reset_index(drop=True)
)

overall_access_rank.index = overall_access_rank.index + 1
overall_access_rank

,city,overall_avg_accessibility
1,Bratislava,45339.809260
2,Turin,6277.231777
3,Brisbane,2995.863780
4,Tallinn,747.484485
5,Singapore,135.026009
6,Copenhagen,58.881015
7,Milan,4.532664


In [4]:
# cities ranked by city similarity
city_similarity_rank = (
    compiled_df[["city", "city_similarity"]]
    .sort_values("city_similarity", ascending=False)
    .reset_index(drop=True)
)

city_similarity_rank.index = city_similarity_rank.index + 1
city_similarity_rank

,city,city_similarity
1,Milan,0.725881
2,Tallinn,0.699522
3,Singapore,0.683946
4,Turin,0.672406
5,Copenhagen,0.642854
6,Brisbane,0.610674
7,Bratislava,0.516961


In [15]:
DATA_DIR = Path("data")
NORMALISATION = 100


def compile_city_rankings(data_dir=DATA_DIR, normalisation=NORMALISATION):
    rows = []

    for fp in sorted(data_dir.glob("*_scores.geojson")):
        city = fp.stem.replace("_scores", "").replace("-", " ").replace("_", " ").title()
        gdf = gpd.read_file(fp)

        gravity_pc_cols = [
            c for c in gdf.columns
            if c.startswith("gravity_") and c.endswith(f"_per_{normalisation}")
        ]

        if not gravity_pc_cols:
            gravity_pc_cols = [
                c for c in gdf.columns
                if c.startswith("gravity_") and "_per_" not in c and c != "avg_gravity"
            ]

        diversity_cols = [
            c for c in ["shannon", "kl_divergence", "jsd", "city_similarity"]
            if c in gdf.columns
        ]

        row = {"city": city}

        cat_means = []
        for col in gravity_pc_cols:
            vals = pd.to_numeric(gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            mean_val = vals.fillna(0).mean()
            row[col] = mean_val
            cat_means.append(mean_val)

        row["overall_avg_accessibility"] = np.mean(cat_means) if cat_means else np.nan

        for col in diversity_cols:
            vals = pd.to_numeric(gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            row[col] = vals.mean()

        rows.append(row)

    compiled_df = pd.DataFrame(rows)
    compiled_df = compiled_df.sort_values("overall_avg_accessibility", ascending=False).reset_index(drop=True)

    return compiled_df


compiled_df = compile_city_rankings()

category_cols = [
    c for c in compiled_df.columns
    if (c.startswith("gravity_") and c.endswith(f"_per_{NORMALISATION}") or c=="overall_avg_accessibility")
]

heatmap_df = compiled_df[["city"] + category_cols].copy()

rename_map = {
    c: c.replace("gravity_", "").replace(f"_per_{NORMALISATION}", "").replace("_", " ").title()
    for c in category_cols
}
heatmap_df = heatmap_df.rename(columns=rename_map)

fig = px.imshow(
    heatmap_df.set_index("city"),
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Reds"
)

fig.update_layout(
    title="Average Per-Capita Accessibility",
    coloraxis_colorbar_title="Avg score"
)

fig.update_layout(
    width=1100,
    height=500,
    font=dict(size=12)
)

fig.update_xaxes(side="bottom")

fig.update_xaxes(title_text="Amenity category")
fig.update_yaxes(title_text="City")

fig.update_traces(
    hovertemplate="City: %{y}<br>Category: %{x}<br>Avg score: %{z:.2f}<extra></extra>"
)

fig.show()

In [9]:
POP_COL = "population"

def compile_population_weighted_accessibility(data_dir=DATA_DIR, pop_col=POP_COL):
    rows = []

    for fp in sorted(data_dir.glob("*_scores.geojson")):
        city = fp.stem.replace("_scores", "").replace("-", " ").replace("_", " ").title()
        gdf = gpd.read_file(fp)

        if pop_col not in gdf.columns:
            print(f"Skipping {city}: no population column found")
            continue

        pop = pd.to_numeric(gdf[pop_col], errors="coerce").fillna(0)

        gravity_cols = [
            c for c in gdf.columns
            if c.startswith("gravity_") and "_per_" not in c and c != "avg_gravity"
        ]

        row = {
            "city": city,
            "population_total": pop.sum()
        }

        weighted_vals = []

        for col in gravity_cols:
            vals = pd.to_numeric(gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

            if pop.sum() > 0:
                w_score = (vals * pop).sum() / pop.sum()
            else:
                w_score = np.nan

            row[col] = w_score
            weighted_vals.append(w_score)

        row["overall_pop_weighted_accessibility"] = np.nanmean(weighted_vals) if weighted_vals else np.nan
        rows.append(row)

    return pd.DataFrame(rows).sort_values(
        "overall_pop_weighted_accessibility",
        ascending=False
    ).reset_index(drop=True)

compiled_pop_weighted = compile_population_weighted_accessibility()
compiled_pop_weighted

,city,population_total,gravity_community,gravity_education,gravity_essential_services,gravity_food_and_drink,gravity_healthcare,gravity_shelter,overall_pop_weighted_accessibility
0,Turin,9.785185e+05,7.995558,15.145381,18.340527,119.532578,21.510028,1.827128,30.725200
1,Singapore,4.421302e+06,10.367072,9.621547,20.134701,111.198690,16.140001,14.998511,30.410087
2,Milan,1.546693e+06,7.325222,11.847745,21.659487,123.000618,14.593317,1.748100,30.029082
3,Copenhagen,3.232337e+06,8.172320,11.493021,6.799518,112.494145,11.077022,3.218952,25.542496
4,Tallinn,4.021769e+05,2.944046,10.828994,10.101653,40.658288,11.897569,25.679374,17.018321
5,Bratislava,7.328475e+05,2.208958,5.021504,6.103011,20.509704,6.431457,1.489717,6.960725
6,Brisbane,9.582931e+05,1.961739,2.555460,2.962271,12.007537,3.176012,5.290028,4.658841


In [10]:
compiled_pop_weighted[["city", "overall_pop_weighted_accessibility"]].sort_values(
    "overall_pop_weighted_accessibility",
    ascending=False
).reset_index(drop=True)

,city,overall_pop_weighted_accessibility
0,Turin,30.725200
1,Singapore,30.410087
2,Milan,30.029082
3,Copenhagen,25.542496
4,Tallinn,17.018321
5,Bratislava,6.960725
6,Brisbane,4.658841


In [14]:
category_cols = [
    c for c in compiled_pop_weighted.columns
    if c.startswith("gravity_") and "_per_" not in c and c != "avg_gravity"
]

heatmap_df = compiled_pop_weighted[["city"] + category_cols + ["overall_pop_weighted_accessibility"]].copy()

rename_map = {
    c: c.replace("gravity_", "").replace("_", " ").title()
    for c in category_cols
}
rename_map["overall_pop_weighted_accessibility"] = "Overall Avg Accessibility"

heatmap_df = heatmap_df.rename(columns=rename_map)

fig = px.imshow(
    heatmap_df.set_index("city"),
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Greens"
)

fig.update_layout(
    title="Population-Weighted Accessibility by Category",
    coloraxis_colorbar_title="Avg score"
)

fig.update_xaxes(title_text="Amenity category")
fig.update_yaxes(title_text="City")

fig.update_traces(
    hovertemplate="City: %{y}<br>Category: %{x}<br>Score: %{z:.2f}<extra></extra>"
)

fig.show()